# 07 · Tuning M2

XGBoost has settings that are chosen, not learned: how many trees, how deep, how fast it learns. This notebook searches them the way Lab 3 does, with `RandomizedSearchCV`, on the **train** split only:

- the folds are **time-ordered** (`TimeSeriesSplit`): each fold trains on earlier hours and scores on the hours right after, so no fold learns from its own future;
- the score is **mean Poisson deviance**, the loss the Poisson objective minimises (lower is better);
- the search space and the number of draws are in config (`tuning.M2`).

Validation is used once, at the end, to compare the best draw with XGBoost's default settings.

In [1]:
import os
import time

import pandas as pd
import xgboost
import yaml
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# run from the repo root, where main.py, config/ and datamart/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from utils.model import LABEL, fairness_groups, feature_columns, load_split, metrics

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
features = feature_columns(config)
# TimeSeriesSplit cuts the rows in order, so sort them by time first
train = load_split("train", features, config).sort_values(["hour_start", "community_area"]).reset_index(drop=True)
validation = load_split("validation", features, config)
tuning = config["tuning"]["M2"]
print(f"train {len(train):,} rows, validation {len(validation):,}; {len(features)} features")
print(f"{tuning['n_iter']} draws x {tuning['cv_folds']} folds = {tuning['n_iter'] * tuning['cv_folds']} fits; space:", tuning["space"])

train 2,533,377 rows, validation 334,411; 19 features
10 draws x 3 folds = 30 fits; space: {'n_estimators': [200, 400, 600], 'max_depth': [3, 4, 6, 8], 'learning_rate': [0.03, 0.05, 0.1], 'min_child_weight': [1, 10, 50], 'subsample': [0.7, 0.85, 1.0]}


## 1 · The folds

Each fold's validation block comes right after its training rows.

In [2]:
folds = TimeSeriesSplit(n_splits=tuning["cv_folds"])
for i, (fit_rows, score_rows) in enumerate(folds.split(train), start=1):
    fit_hours, score_hours = train.loc[fit_rows, "hour_start"], train.loc[score_rows, "hour_start"]
    print(f"fold {i}: fit {fit_hours.min():%Y-%m} to {fit_hours.max():%Y-%m} ({len(fit_rows):,} rows), "
          f"score {score_hours.min():%Y-%m} to {score_hours.max():%Y-%m} ({len(score_rows):,})")

fold 1: fit 2018-04 to 2019-03 (633,345 rows), score 2019-03 to 2020-02 (633,344)
fold 2: fit 2018-04 to 2020-02 (1,266,689 rows), score 2020-02 to 2021-01 (633,344)
fold 3: fit 2018-04 to 2021-01 (1,900,033 rows), score 2021-01 to 2021-12 (633,344)


## 2 · The search

In [3]:
search = RandomizedSearchCV(
    xgboost.XGBRegressor(objective="count:poisson", tree_method="hist", random_state=0),
    param_distributions=tuning["space"],
    n_iter=tuning["n_iter"],
    cv=folds,
    scoring="neg_mean_poisson_deviance",  # sklearn maximises scores, so the deviance comes negated
    random_state=0,
)
started = time.time()
search.fit(train[features], train[LABEL])
print(f"{time.time() - started:.0f} s; best deviance {-search.best_score_:.4f} with {search.best_params_}")

496 s; best deviance 0.5443 with {'subsample': 0.7, 'n_estimators': 600, 'min_child_weight': 50, 'max_depth': 4, 'learning_rate': 0.05}


## 3 · Every draw, best first

In [4]:
draws = pd.DataFrame(search.cv_results_)
draws["deviance"] = -draws["mean_test_score"]
draws["sd"] = draws["std_test_score"]
draws[["deviance", "sd"] + [f"param_{p}" for p in tuning["space"]]].sort_values("deviance").round(4)

,deviance,sd,param_n_estimators,param_max_depth,param_learning_rate,param_min_child_weight,param_subsample
9,0.5443,0.0318,600,4,0.05,50,0.70
3,0.5446,0.0309,400,6,0.10,1,0.85
4,0.5448,0.0318,400,4,0.05,1,1.00
0,0.5451,0.0307,600,6,0.10,50,0.85
7,0.5457,0.0319,200,3,0.10,50,0.70
2,0.5459,0.0318,600,3,0.03,10,0.70
6,0.5460,0.0312,200,8,0.03,50,1.00
5,0.5461,0.0318,200,4,0.05,50,0.70
8,0.5464,0.0314,200,6,0.03,10,0.70
1,0.5465,0.0315,200,6,0.03,10,1.00


## 4 · The best draw against XGBoost's defaults, on validation

Both are refit on all of train, then scored once on validation with the training job's metrics.

In [5]:
groups = fairness_groups(train, config)
rows = []
for name, params in [("defaults", {}), ("tuned", search.best_params_)]:
    model = xgboost.XGBRegressor(objective="count:poisson", tree_method="hist", random_state=0, **params)
    model.fit(train[features], train[LABEL])
    scored = pd.DataFrame({"community_area": validation["community_area"], "hour_start": validation["hour_start"],
                           "actual": validation[LABEL], "predicted": model.predict(validation[features])})
    m = metrics(scored, config, groups)
    rows.append({"M2": name, "P0": m["P0"], "MAE": m["MAE"], "deviance": m["poisson_deviance"], "bias": m["bias"]})
pd.DataFrame(rows).set_index("M2").round(4)

,P0,MAE,deviance,bias
M2,,,,
defaults,0.3019,0.2446,0.5514,1.0185
tuned,0.3017,0.2438,0.5518,1.0049
